# 02 | Primary Key and Identifier Analysis

**Purpose:** Profile candidate identifiers across the Olist tables by checking row counts, unique values, nulls, duplicates, and uniqueness percentages.

**Inputs:** The nine original Olist CSV files in `data/raw/`.

**Interpretation:** This is an identifier-profiling exercise. A high uniqueness percentage is evidence for evaluating a key candidate, but does not by itself prove that a column is a valid primary key. Composite keys and table grain must also be considered.

Run after `01_dataset_audit.ipynb`.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = Path(os.environ.get("OLIST_DATA_DIR", PROJECT_ROOT / "data" / "raw")).expanduser()

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Olist CSV folder not found: {DATA_DIR}\n"
        "Download the Olist CSV files and place them in data/raw, "
        "or set the OLIST_DATA_DIR environment variable."
    )

print("Dataset folder:", DATA_DIR)


## Load all nine source tables


In [ ]:
datasets = {}

for file in DATA_DIR.glob("*.csv"):

    dataset_name = file.stem

    datasets[dataset_name] = pd.read_csv(
        file,
        low_memory=False
    )

print("Total datasets loaded:", len(datasets))

for name, df in datasets.items():
    print(f"{name}: {df.shape[0]:,} rows × {df.shape[1]} columns")

if len(datasets) != 9:
    raise ValueError("Expected 9 CSV files. Check your dataset folder.")

## Define identifier candidates


In [ ]:
key_candidates = {

    "olist_orders_dataset": [
        "order_id",
        "customer_id"
    ],

    "olist_customers_dataset": [
        "customer_id",
        "customer_unique_id"
    ],

    "olist_order_items_dataset": [
        "order_id",
        "order_item_id",
        "product_id",
        "seller_id"
    ],

    "olist_order_payments_dataset": [
        "order_id",
        "payment_sequential"
    ],

    "olist_order_reviews_dataset": [
        "review_id",
        "order_id"
    ],

    "olist_products_dataset": [
        "product_id"
    ],

    "olist_sellers_dataset": [
        "seller_id"
    ],

    "product_category_name_translation": [
        "product_category_name",
        "product_category_name_english"
    ],

    "olist_geolocation_dataset": [
        "geolocation_zip_code_prefix"
    ]
}

print("Identifier candidates defined successfully!")

## Calculate identifier quality metrics


In [ ]:
identifier_results = []

for table, columns in key_candidates.items():

    df = datasets[table]

    for col in columns:

        if col in df.columns:

            total_records = len(df)

            null_count = df[col].isnull().sum()

            unique_count = df[col].nunique()

            duplicate_count = df[col].duplicated().sum()

            identifier_results.append({

                "Dataset": table,

                "Column": col,

                "Total Records": total_records,

                "Unique Values": unique_count,

                "Null Values": null_count,

                "Duplicate Values": duplicate_count,

                "Uniqueness %": round(
                    (unique_count / total_records) * 100, 2
                ) if total_records else 0

            })

identifier_df = pd.DataFrame(identifier_results)

display(identifier_df)

## Validate composite identifiers

Some source tables are defined by a combination of columns rather than a single identifier. Test the candidate pairs below at their expected table grain.

In [ ]:
composite_key_candidates = {
    "olist_order_items_dataset": ["order_id", "order_item_id"],
    "olist_order_payments_dataset": ["order_id", "payment_sequential"],
}

composite_results = []

for table, columns in composite_key_candidates.items():
    df = datasets[table]
    missing_columns = [column for column in columns if column not in df.columns]
    if missing_columns:
        raise KeyError(f"{table} is missing expected columns: {missing_columns}")

    total_rows = len(df)
    null_key_rows = int(df[columns].isna().any(axis=1).sum())
    duplicate_key_rows = int(df.duplicated(subset=columns, keep="first").sum())
    unique_combinations = int(df[columns].drop_duplicates().shape[0])

    composite_results.append({
        "Dataset": table,
        "Columns": " + ".join(columns),
        "Total Rows": total_rows,
        "Unique Combinations": unique_combinations,
        "Rows with Null Key Part": null_key_rows,
        "Duplicate Composite-Key Rows": duplicate_key_rows,
        "Unique Combination %": round(unique_combinations / total_rows * 100, 2) if total_rows else 0,
    })

composite_key_df = pd.DataFrame(composite_results)
display(composite_key_df)